# Credit Default Risk Scorecard
## Notebook 2: Feature Engineering & Model
**Author:** Simpson Gundlapally
**ROC-AUC Achieved:** 0.8338

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded')

## Step 1 — Load & Clean

In [ ]:
df = pd.read_csv('../data/cs-training.csv', index_col=0)

# Handle missing values
df['MonthlyIncome']      = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)

# Remove underage (data error)
df = df[df['age'] >= 18].copy()

# Cap outliers
df['RevolvingUtilizationOfUnsecuredLines'] = df['RevolvingUtilizationOfUnsecuredLines'].clip(0, 1)
df['DebtRatio'] = df['DebtRatio'].clip(0, 10)

print(f'Clean dataset: {len(df):,} rows')
print(f'Default rate: {df["SeriousDlqin2yrs"].mean():.2%}')

## Step 2 — Feature Engineering

In [ ]:
# Combined late payment score
df['TotalLatePays'] = (
    df['NumberOfTime30-59DaysPastDueNotWorse'] +
    df['NumberOfTime60-89DaysPastDueNotWorse'] +
    df['NumberOfTimes90DaysLate']
)

# Debt burden
df['DebtToIncomeRatio'] = df['DebtRatio'] * df['MonthlyIncome']

# Credit density — how many lines relative to age
df['CreditDensity'] = df['NumberOfOpenCreditLinesAndLoans'] / (df['age'] - 17 + 0.1)

print('New features created:')
print(f'  TotalLatePays    — mean: {df["TotalLatePays"].mean():.2f}')
print(f'  DebtToIncomeRatio — mean: {df["DebtToIncomeRatio"].mean():.2f}')
print(f'  CreditDensity    — mean: {df["CreditDensity"].mean():.2f}')

## Step 3 — Train Model

In [ ]:
features = [
    'RevolvingUtilizationOfUnsecuredLines', 'age',
    'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio',
    'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans',
    'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines',
    'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents',
    'TotalLatePays', 'DebtToIncomeRatio', 'CreditDensity'
]

X = df[features]
y = df['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# class_weight='balanced' handles the 1:14 class imbalance
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)

y_pred  = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:, 1]
auc     = roc_auc_score(y_test, y_proba)

print(f'ROC-AUC Score: {auc:.4f}')
print(classification_report(y_test, y_pred))

## Step 4 — Build Scorecard (300-850 Scale)

In [ ]:
def prob_to_score(prob):
    """Convert default probability to credit score (300-850 FICO-style)"""
    factor = 20 / np.log(2)
    offset = 600 - factor * np.log(50)
    score  = offset - factor * np.log(np.clip(prob / (1 - prob + 1e-10), 1e-10, None))
    return np.clip(score, 300, 850).astype(int)

results = X_test.copy()
results['ActualDefault']      = y_test.values
results['DefaultProbability'] = y_proba
results['CreditScore']        = prob_to_score(y_proba)
results['Decision'] = pd.cut(results['CreditScore'],
    bins=[299, 500, 600, 700, 850],
    labels=['DECLINE', 'MANUAL REVIEW', 'CONDITIONAL APPROVE', 'APPROVE'])

band_summary = results.groupby('Decision', observed=True).agg(
    Applicants   = ('ActualDefault', 'count'),
    Defaults     = ('ActualDefault', 'sum'),
    DefaultRate  = ('ActualDefault', 'mean'),
    AvgScore     = ('CreditScore',   'mean')
).reset_index()
band_summary['DefaultRate'] = (band_summary['DefaultRate'] * 100).round(1)
band_summary['AvgScore']    = band_summary['AvgScore'].round(0).astype(int)

print('\nCREDIT SCORECARD — DECISION BANDS')
print('='*60)
print(band_summary.to_string(index=False))